# Faz 7 — Qwen3-VL gerçek embedding üretimi
Bu notebook yalnızca 2048d gerçek vektör üretir. GPU yoksa ilk hücrede güvenli biçimde durur.

In [ ]:
import subprocess
probe = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'], capture_output=True, text=True)
if probe.returncode != 0:
    raise SystemExit('GPU bulunamadı. Colab > Runtime > Change runtime type > GPU seçip tekrar çalıştırın.')
print(probe.stdout)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.3.2', 'pandas==2.3.1', 'pyarrow==21.0.0', 'huggingface-hub==0.34.3', 'transformers==4.54.1', 'accelerate==1.9.0', 'qwen-vl-utils==0.0.11'], check=True)
from pathlib import Path
if not Path('/content/Qwen3-VL-Embedding').exists():
    subprocess.run(['git', 'clone', 'https://github.com/QwenLM/Qwen3-VL-Embedding.git', '/content/Qwen3-VL-Embedding'], check=True)
sys.path.insert(0, '/content/Qwen3-VL-Embedding')

In [ ]:
from huggingface_hub import model_info, snapshot_download
MODEL_ID = 'Qwen/Qwen3-VL-Embedding-2B'
MODEL_REVISION = model_info(MODEL_ID).sha
MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir='/content/models/qwen3-vl-embedding-2b')
print(MODEL_PATH)

In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
if major >= 8:
    TORCH_DTYPE, ATTN_IMPL = torch.bfloat16, 'flash_attention_2'
else:
    TORCH_DTYPE, ATTN_IMPL = torch.float16, 'sdpa'
if major >= 8:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'flash-attn==2.8.3', '--no-build-isolation'], check=True)
print({'gpu': gpu_name, 'capability': [major, minor], 'dtype': str(TORCH_DTYPE), 'attention': ATTN_IMPL})

In [ ]:
from pathlib import Path
import pandas as pd
DATASET = 'auair'  # capera / seadronessee / auair
FRAMES_PER_ITEM = 8
CHECKPOINT_EVERY = 200
BATCH_SIZE = 1
INPUT_PARQUET = Path(f'/content/drive/MyDrive/mvi/{DATASET}_segments.parquet')
FRAME_ROOT = Path(f'/content/drive/MyDrive/mvi/{DATASET}_frames')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mvi/artifacts/embeddings')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
segments = pd.read_parquet(INPUT_PARQUET)
id_col = 'segment_id' if 'segment_id' in segments else 'id'
print(DATASET, len(segments), segments.columns.tolist())

In [ ]:
import json, time
import numpy as np
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder
model = Qwen3VLEmbedder(model_name_or_path=MODEL_PATH, torch_dtype=TORCH_DTYPE, attn_implementation=ATTN_IMPL)
partial_path = OUTPUT_ROOT / f'{DATASET}_2048.partial.npy'
done_path = OUTPUT_ROOT / f'{DATASET}_done_ids.json'
errors_path = OUTPUT_ROOT / f'{DATASET}_errors.jsonl'
done = set(json.loads(done_path.read_text())) if done_path.exists() else set()
vectors = np.lib.format.open_memmap(partial_path, mode='r+' if partial_path.exists() else 'w+', dtype=np.float32, shape=(len(segments), 2048))
started = time.time()
for idx, row in segments.iterrows():
    segment_id = str(row[id_col])
    if segment_id in done:
        continue
    try:
        frame_names = list(row.get('frame_names', []))[:FRAMES_PER_ITEM]
        frame_paths = [str(FRAME_ROOT / name) for name in frame_names]
        source = frame_paths if frame_paths else str(row.get('file_path', ''))
        output = model.process([{'video': source}])
        vector = output[0].detach().cpu().float().numpy() if hasattr(output[0], 'detach') else np.asarray(output[0], dtype=np.float32)
        vector = vector.astype(np.float32); vector /= np.linalg.norm(vector)
        vectors[idx] = vector; done.add(segment_id)
    except Exception as exc:
        with errors_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps({'segment_id': segment_id, 'error': str(exc)}, ensure_ascii=False) + '\n')
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        vectors.flush(); done_path.write_text(json.dumps(sorted(done)), encoding='utf-8'); print(idx + 1, '/', len(segments))
vectors.flush(); done_path.write_text(json.dumps(sorted(done)), encoding='utf-8')
ELAPSED_S = time.time() - started

In [ ]:
final = np.asarray(vectors, dtype=np.float32)
assert final.shape == (len(segments), 2048)
assert np.isfinite(final).all()
np.testing.assert_allclose(np.linalg.norm(final, axis=1), 1.0, atol=1e-5)
final_path = OUTPUT_ROOT / f'{DATASET}_2048.npy'
np.save(final_path, final)
segments[[id_col]].rename(columns={id_col: 'segment_id'}).to_parquet(OUTPUT_ROOT / f'{DATASET}_ids.parquet', index=False)
manifest = {'dataset': DATASET, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'dtype': str(TORCH_DTYPE), 'attention': ATTN_IMPL, 'gpu': gpu_name, 'frames_per_item': FRAMES_PER_ITEM, 'elapsed_s': ELAPSED_S, 'count': len(final), 'dimension': 2048}
(OUTPUT_ROOT / 'embedding_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_ROOT / f'embeddings_{DATASET}'), 'zip', OUTPUT_ROOT, '.', logger=None)
print('Drive ZIP:', archive)
from google.colab import files
files.download(archive)